In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.product-placement",
        description="Identify brand product placement in the video footage",
        worker_release="qwen3-instruct",
        text_prompt="""
          You are an expert product placement auditor reviewing video footage. Your task is to determine if a specific target product is visible in this video segment.

          TARGET PRODUCT: A bright neon-blue aluminum soda can featuring a bold yellow lightning-bolt logo on the front.

          Analyze the frames carefully and output ONLY one of the following exact labels:
          product: The target neon-blue can with the yellow lightning bolt is clearly visible and identifiable in the frame. It may be resting on a surface, being held, or being actively consumed by a person.
          non_target: The target product is completely absent, blocked from view, or so heavily blurred/distorted that its specific design cannot be confidently identified.

          CRITICAL: You must not output anything other than exactly "product" or "non_target". Do not include punctuation, markdown formatting, or explanations. Do not guess or assume the product is there based on context; you must physically see the specific product.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=10,
            image_size=512
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.product-placement:latest",
   )
])

video_path = ... # Add path to video

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path(video_path)
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")